In [2]:
# Step 1: Load the baseline dataset
import pandas as pd

df = pd.read_csv('alphatrade_phase1.csv')

print(df.shape)
df.head()

(1508, 19)


,Date,Close,High,Low,Open,Volume,SMA_20,SMA_50,EMA_20,EMA_50,Daily_Return,Volatility,RSI,MACD,Signal_Line,BB_Middle,BB_Upper,BB_Lower,Signal
0,2020-01-02,72.333878,72.394086,71.091184,71.344054,135480400,NaN,NaN,72.333878,72.333878,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,0
1,2020-01-03,71.630630,72.389250,71.406659,71.563198,146322800,NaN,NaN,72.266902,72.306299,-0.009722,NaN,NaN,-0.056099,-0.011220,NaN,NaN,NaN,0
2,2020-01-06,72.201424,72.239958,70.503561,70.754028,118387200,NaN,NaN,72.260666,72.302186,0.007969,NaN,NaN,-0.053879,-0.019752,NaN,NaN,NaN,0
3,2020-01-07,71.861847,72.466330,71.642689,72.211049,108872000,NaN,NaN,72.222683,72.284918,-0.004703,NaN,NaN,-0.078615,-0.031524,NaN,NaN,NaN,0
4,2020-01-08,73.017853,73.318893,71.565636,71.565636,132079200,NaN,NaN,72.298413,72.313661,0.016087,NaN,NaN,-0.004881,-0.026196,NaN,NaN,NaN,0


In [3]:
'''Step 2: Create Lag Features
Concept
These tell the model what happened in previous days.'''

df['Lag_1'] = df['Close'].shift(1)
df['Lag_2'] = df['Close'].shift(2)
df['Lag_3'] = df['Close'].shift(3)
df['Lag_5'] = df['Close'].shift(5)

In [4]:
'''Step 3: Create Momentum Features
Concept
These measure how much the stock has moved.'''

df['Momentum_5'] = df['Close'] - df['Close'].shift(5)
df['Momentum_10'] = df['Close'] - df['Close'].shift(10)

In [5]:
'''Step 4: Create Rolling Statistics
Concept
These capture short-term trend and volatility.'''

df['Rolling_Mean_5'] = df['Close'].rolling(5).mean()

df['Rolling_STD_5'] = df['Close'].rolling(5).std()

In [6]:
'''Step 5: Create Percentage Returns
Concept
These measure percentage movement over time.'''

df['Return_5'] = df['Close'].pct_change(5)

df['Return_10'] = df['Close'].pct_change(10)

In [7]:
# Step 6: Remove the newly created NaN values
df = df.dropna()

In [8]:
# Step 7: Verify
print(df.shape)
print(df.isnull().sum().sum())
print(df.columns)

(1459, 29)
0
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA_20', 'SMA_50',
       'EMA_20', 'EMA_50', 'Daily_Return', 'Volatility', 'RSI', 'MACD',
       'Signal_Line', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Signal', 'Lag_1',
       'Lag_2', 'Lag_3', 'Lag_5', 'Momentum_5', 'Momentum_10',
       'Rolling_Mean_5', 'Rolling_STD_5', 'Return_5', 'Return_10'],
      dtype='str')


In [9]:
print(df.shape)
print(df.columns)

(1459, 29)
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA_20', 'SMA_50',
       'EMA_20', 'EMA_50', 'Daily_Return', 'Volatility', 'RSI', 'MACD',
       'Signal_Line', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Signal', 'Lag_1',
       'Lag_2', 'Lag_3', 'Lag_5', 'Momentum_5', 'Momentum_10',
       'Rolling_Mean_5', 'Rolling_STD_5', 'Return_5', 'Return_10'],
      dtype='str')


In [10]:
'''Step 1: Create the new feature list
This time, include both the old and advanced features:'''
features = [
    # Original features
    'Close',
    'Volume',
    'SMA_20',
    'SMA_50',
    'EMA_20',
    'EMA_50',
    'Daily_Return',
    'Volatility',
    'RSI',
    'MACD',
    'Signal_Line',

    # Lag features
    'Lag_1',
    'Lag_2',
    'Lag_3',
    'Lag_5',

    # Momentum features
    'Momentum_5',
    'Momentum_10',

    # Rolling statistics
    'Rolling_Mean_5',
    'Rolling_STD_5',

    # Return features
    'Return_5',
    'Return_10'
]

In [13]:
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

df = df[:-1]

In [14]:
df = df.dropna()

In [15]:
print(df['Target'].value_counts())
print(df.shape)

Target
1    778
0    680
Name: count, dtype: int64
(1458, 30)


In [ ]:
X = df[features]
y = df['Target']

print(X.shape)
print(y.shape)

(1458, 21)
(1458,)


In [19]:
# Chronological Split 
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(1166, 21)
(292, 21)
(1166,)
(292,)


In [20]:
# Then train Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_pred))

Random Forest Accuracy: 0.5


In [21]:
# Then train XGBoost
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("XGBoost Accuracy:",
      accuracy_score(y_test, xgb_pred))

XGBoost Accuracy: 0.5102739726027398


In [22]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False)

print(importance)

           Feature  Importance
6     Daily_Return    0.063721
18   Rolling_STD_5    0.060503
7       Volatility    0.059872
1           Volume    0.055986
8              RSI    0.055445
16     Momentum_10    0.054465
20       Return_10    0.053192
15      Momentum_5    0.051196
10     Signal_Line    0.050773
19        Return_5    0.049699
9             MACD    0.047111
14           Lag_5    0.043596
12           Lag_2    0.041692
0            Close    0.041616
5           EMA_50    0.041432
11           Lag_1    0.040752
3           SMA_50    0.040734
13           Lag_3    0.040030
4           EMA_20    0.038341
2           SMA_20    0.035710
17  Rolling_Mean_5    0.034135


In [23]:
# Feature Pruning:
best_features = [
    'Daily_Return',
    'Rolling_STD_5',
    'Volatility',
    'Volume',
    'RSI',
    'Momentum_10',
    'Return_10',
    'Momentum_5',
    'Signal_Line',
    'Return_5',
    'MACD'
]

In [24]:
X = df[best_features]
y = df['Target']

print(X.shape)
print(y.shape)

(1458, 11)
(1458,)


In [25]:
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [26]:
print(X_train.shape)
print(X_test.shape)

(1166, 11)
(292, 11)


In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_pred))

Random Forest Accuracy: 0.4897260273972603


In [28]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("XGBoost Accuracy:",
      accuracy_score(y_test, xgb_pred))

XGBoost Accuracy: 0.5034246575342466
